In [1]:
import os
import glob
import random
import json
from collections import Counter

os.environ["SETUPTOOLS_USE_DISTUTILS"] = "stdlib"
!pip install scikit-learn matplotlib -q

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt

# no viene con sklearn, la necesitamos para el split estratificado multi-label
!pip install iterative-stratification -q
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
LOCAL_DATA_DIR = "data_deteccion"
os.makedirs(f"{LOCAL_DATA_DIR}/images", exist_ok=True)
os.makedirs(f"{LOCAL_DATA_DIR}/labels", exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_classification"

import subprocess
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/images/", f"{LOCAL_DATA_DIR}/images/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/labels/", f"{LOCAL_DATA_DIR}/labels/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/classes.txt", f"{LOCAL_DATA_DIR}/classes.txt", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/data.yaml", f"{LOCAL_DATA_DIR}/data.yaml", "--quiet"], check=True)

n_images = len(os.listdir(f"{LOCAL_DATA_DIR}/images"))
n_labels = len(os.listdir(f"{LOCAL_DATA_DIR}/labels"))
print(f"Imagenes: {n_images} | Labels: {n_labels}")

In [ ]:
# el orden de esta lista tiene que coincidir con las lineas de classes.txt
with open(f"{LOCAL_DATA_DIR}/classes.txt") as f:
    all_classes = [line.strip() for line in f if line.strip()]

FINAL_CLASSES = [
    "Diagonal crack",
    "Concrete spalling",
    "Corrosion",
    "Shear crack",
    "Flexural/rocking crack",
    "Debris",
    "Detachment crack",
]

orig_id_to_name = {i: name for i, name in enumerate(all_classes)}
name_to_final_idx = {name: idx for idx, name in enumerate(FINAL_CLASSES)}

print(f"{len(all_classes)} clases en el dataset original, nos quedamos con {len(FINAL_CLASSES)}")

In [ ]:
# conteo de imagenes por clase para TODAS las clases del dataset original (classes.txt),
# no solo las FINAL_CLASSES. es multi-label: una imagen puede aparecer en varias clases.
conteo_todas = Counter()
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    with open(label_path) as f:
        clases_en_imagen = {int(l.split()[0]) for l in f if l.strip()}
    for class_id in clases_en_imagen:
        conteo_todas[orig_id_to_name[class_id]] += 1

resumen_todas = pd.DataFrame({
    "imagenes": [conteo_todas.get(name, 0) for name in all_classes],
    "seleccionada": [name in name_to_final_idx for name in all_classes],
}, index=all_classes).sort_values("imagenes", ascending=False)

print(f"total de clases en el dataset original: {len(all_classes)}\n")
print(resumen_todas)


In [ ]:
label_files = sorted(glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"))

rows = []
descartadas = 0

for label_path in label_files:
    img_name = os.path.splitext(os.path.basename(label_path))[0]

    # las imagenes no siempre vienen con la misma extension, probamos las mas comunes
    img_path = None
    for ext in (".jpg", ".jpeg", ".png"):
        candidate = f"{LOCAL_DATA_DIR}/images/{img_name}{ext}"
        if os.path.exists(candidate):
            img_path = candidate
            break
    if img_path is None:
        continue

    with open(label_path) as f:
        lines = [l.strip() for l in f if l.strip()]

    presentes = set()
    for line in lines:
        class_id = int(line.split()[0])
        class_name = orig_id_to_name[class_id]
        if class_name in name_to_final_idx:
            presentes.add(name_to_final_idx[class_name])

    if not presentes:
        # tiene daño pero de una clase que no vamos a modelar, la sacamos del dataset
        descartadas += 1
        continue

    multi_hot = np.zeros(len(FINAL_CLASSES), dtype=int)
    for idx in presentes:
        multi_hot[idx] = 1

    rows.append([img_path] + multi_hot.tolist())

df = pd.DataFrame(rows, columns=["image_path"] + FINAL_CLASSES)
print(f"quedaron {len(df)} imagenes, se descartaron {descartadas} por tener solo clases fuera de las finales")
df.head()

In [ ]:
conteo = df[FINAL_CLASSES].sum().sort_values(ascending=False)
print(conteo)

conteo.plot(kind="bar", figsize=(8, 4))
plt.ylabel("cantidad de imagenes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# cuantos daños distintos aparecen juntos en una misma imagen
n_danios = df[FINAL_CLASSES].sum(axis=1)
print(n_danios.value_counts().sort_index())

In [ ]:
# conteo de imagenes por clase
# ojo: es multi-label, una imagen puede contar en varias clases a la vez,
# por eso la suma de la columna "imagenes" es mayor que el total del dataset
resumen = pd.DataFrame({
    "imagenes": df[FINAL_CLASSES].sum(),
    "% del dataset": (df[FINAL_CLASSES].mean() * 100).round(1),
    # imagenes donde ese daño es el unico presente
    "solo esta clase": [
        int(((df[cls] == 1) & (df[FINAL_CLASSES].sum(axis=1) == 1)).sum())
        for cls in FINAL_CLASSES
    ],
}).sort_values("imagenes", ascending=False)

print(f"total de imagenes en el dataset: {len(df)}\n")
print(resumen)
print(f"\nclase mas frecuente: {resumen.index[0]} ({resumen['imagenes'].iloc[0]})")
print(f"clase menos frecuente: {resumen.index[-1]} ({resumen['imagenes'].iloc[-1]})")
print(f"desbalance (mayor/menor): {resumen['imagenes'].iloc[0] / max(resumen['imagenes'].iloc[-1], 1):.1f}x")


In [ ]:
X = df["image_path"].values.reshape(-1, 1)
y = df[FINAL_CLASSES].values

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(msss.split(X, y))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"train: {len(train_df)} | val: {len(val_df)}")
print(train_df[FINAL_CLASSES].sum())
print(val_df[FINAL_CLASSES].sum())

In [ ]:
# oversampling: duplicamos ejemplos de las clases con menos imagenes hasta acercarnos
# a la clase mas grande. como una imagen puede tener mas de un daño, esto de paso
# empuja un poco a las clases que suelen aparecer junto a la minoritaria
target = train_df[FINAL_CLASSES].sum().max()

partes = [train_df]
for cls in FINAL_CLASSES:
    actual = train_df[cls].sum()
    faltante = target - actual
    if faltante <= 0:
        continue
    candidatos = train_df[train_df[cls] == 1]
    extra = candidatos.sample(n=int(faltante), replace=True, random_state=42)
    partes.append(extra)

train_df_balanced = pd.concat(partes, ignore_index=True)
train_df_balanced = train_df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"train antes del oversampling: {len(train_df)}, despues: {len(train_df_balanced)}")
print(train_df_balanced[FINAL_CLASSES].sum())

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

# el flip horizontal deberia estar bien para grietas y daños estructurales en general,
# pero si en algun momento notas que la orientacion importa para alguna clase, sacalo
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
])

def make_dataset(dataframe, training):
    paths = dataframe["image_path"].values
    labels = dataframe[FINAL_CLASSES].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2000, seed=42)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df_balanced, training=True)
val_ds = make_dataset(val_df, training=False)

In [ ]:
base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)

# sigmoid, no softmax: cada daño es independiente, una imagen puede tener varios a la vez
outputs = layers.Dense(len(FINAL_CLASSES), activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="auc", multi_label=True),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)
model.summary()

In [ ]:
# esto imprime precision/recall/f1 de cada tipo de daño al final de cada epoca,
# no solo el promedio general que muestra el fit() por defecto
class MetricasPorDano(tf.keras.callbacks.Callback):
    def __init__(self, val_dataset, class_names, threshold=0.5):
        super().__init__()
        self.val_dataset = val_dataset
        self.class_names = class_names
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        y_true = np.concatenate([y.numpy() for _, y in self.val_dataset], axis=0)
        y_probs = self.model.predict(self.val_dataset, verbose=0)
        y_pred = (y_probs > self.threshold).astype(int)

        print(f"\nmetricas por clase (umbral {self.threshold}) - fin de epoca {epoch + 1}")
        for i, cls in enumerate(self.class_names):
            p = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
            r = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)
            f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)
            print(f"  {cls:28s} precision {p:.2f}  recall {r:.2f}  f1 {f1:.2f}")

metricas_callback = MetricasPorDano(val_ds, FINAL_CLASSES)

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
    ModelCheckpoint("modelo_danos.keras", monitor="val_loss", save_best_only=True),
    metricas_callback,
]

# fase 1: la base va congelada, solo entrena la cabeza nueva
history_head = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

In [ ]:
# fase 2: descongelamos el ultimo 20% de EfficientNet y afinamos con LR bajo
base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.8)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="auc", multi_label=True),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

In [ ]:
val_probs = model.predict(val_ds)
val_true = val_df[FINAL_CLASSES].values

# con clases desbalanceadas el umbral 0.5 casi nunca es el mejor, probamos varios
# por clase y nos quedamos con el que da mejor f1 en validacion
best_thresholds = {}
for i, cls in enumerate(FINAL_CLASSES):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 0.9, 0.05):
        preds = (val_probs[:, i] > t).astype(int)
        f1 = f1_score(val_true[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[cls] = round(float(best_t), 2)
    print(f"{cls}: umbral {best_t:.2f}, f1 {best_f1:.3f}")

val_preds_tuned = np.array([
    (val_probs[:, i] > best_thresholds[cls]).astype(int)
    for i, cls in enumerate(FINAL_CLASSES)
]).T

print(classification_report(val_true, val_preds_tuned, target_names=FINAL_CLASSES, zero_division=0))

In [ ]:
def predecir_danios(ruta_imagen, umbral_por_clase=best_thresholds):
    img = tf.io.read_file(ruta_imagen)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.expand_dims(img, 0)

    probs = model.predict(img, verbose=0)[0]
    detectados = [
        (FINAL_CLASSES[i], float(probs[i]))
        for i in range(len(FINAL_CLASSES))
        if probs[i] > umbral_por_clase[FINAL_CLASSES[i]]
    ]
    detectados.sort(key=lambda x: x[1], reverse=True)
    return detectados

# predecir_danios("data_deteccion/images/alguna_imagen.jpg")

In [ ]:
model.save("modelo_danos_estructurales.keras")

with open("thresholds.json", "w") as f:
    json.dump(best_thresholds, f, indent=2)

print("modelo y thresholds guardados")